# 102장 · POST 요청·응답 계약과 저장 상태

API의 요청·응답 계약에는 성공만 담지 않습니다. 답은 만들었지만 저장하지 못한 경우를 호출자가 알아볼 수 있도록 결과를 나눕니다.

조회 함수를 호출하는 API 요청에 세션 ID와 요청 ID를 포함합니다. 반환된 업무 상태, 저장 결과, 출처가 화면에서 사용하는 필드와 일치하는지 대조합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 팀의 조회 기록을 응답 계약에 붙이기

101장의 선택을 응답 본문으로 옮깁니다. HTTP 전송 상태와 질문 자체의 처리 결과를 따로 적어, 전송이 성공했어도 답할 근거가 없는 경우를 표현합니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter101.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter102-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter101.json"
sample_file = here / "chapter102-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['adapter'])

교재 예시 chapter102-start.json
공개 문서 조회


두 번째 줄은 읽은 파일의 `adapter` 값입니다.

## POST /p1/ask의 JSON 요청

준비 계약은 session_id·request_id·question 세 문자열을 받습니다. 수업 주체는 준비한 헤더나 서버의 검사 설정으로 결정합니다. 질문 본문에 적은 권한을 서버가 믿어서는 안 됩니다. 유효한 대화를 먼저 만든 뒤 다음 요청을 보낸다는 설정입니다.


### 셀 01 · POST 요청 JSON 직렬화

준비된 정상 사례의 대화 lesson, 요청 normal에 정확히 어떤 문장이 실리는지 확인합니다.


In [1]:
import json
request = {"session_id": "lesson", "request_id": "normal",
           "question": "장비 A의 점검 전 확인 사항은?"}
print(json.dumps(request, ensure_ascii=False))

{"session_id": "lesson", "request_id": "normal", "question": "장비 A의 점검 전 확인 사항은?"}


세 필드가 한 요청으로 묶입니다. session_id는 대화를 찾고 request_id는 해당 시도를 추적합니다. 이 셀은 전송 전의 JSON을 만들었습니다.


### 셀 02 · 응답의 출처 필드

로컬 전송 기록 정상 행에서 화면이 읽을 필드만 발췌했습니다. 실제 응답 원본에는 job 등 추가 필드도 있습니다.


In [2]:
transport_row = {'status_code': 200,
 'body': {'contract_version': 'P1-H1-prepared-v1',
          'result_kind': 'P1',
          'python_response': {'request_id': 'normal',
                              'session_id': 'lesson',
                              'boundary_status': 'ok',
                              'workflow_status': 'completed',
                              'result': {'status': 'answered',
                                         'answer': '점검 전 전원 표시등을 확인합니다.',
                                         'sources': [{'id': 'RULE-A',
                                                      'text': '점검 전 전원 '
                                                              '표시등을 확인합니다.',
                                                      'source': '가상 장비실 규정 '
                                                                'A',
                                                      'page': 1,
                                                      'section': '2.1',
                                                      'revised_on': '2026-11-01',
                                                      'version': 'rules-v1'}],
                                         'mode': 'local_rule_fixture',
                                         'model_calls': 0},
                              'storage_status': 'committed',
                              'error': None}}}
payload = transport_row["body"]["python_response"]
print(transport_row["status_code"], payload["result"]["status"], payload["storage_status"])
print(payload["result"]["sources"][0]["id"])

200 answered committed
RULE-A


기록에는 200·answered·committed와 RULE-A가 있습니다. 200은 준비 코드의 전송 상태, answered는 질문 처리 결과, committed는 저장 결과입니다. 어느 하나만 보고 나머지를 추정하지 않습니다.

200은 요청이 처리됐다는 HTTP 상태이고 committed는 서비스가 기록한 저장 결과입니다. 사용자가 다음 접속에서도 답을 볼 수 있어야 한다면 같은 요청 ID로 저장 기록을 다시 읽는 검사가 필요합니다. 답변 생성과 저장 성공을 구분할 수 있는 응답 예입니다.


#### 응답 상태가 200일 때의 후속 확인

<mark>HTTP 응답, 답변 생성, 저장 완료는 서로 다른 지점에서 확인합니다.</mark> 같은 요청 ID를 따라 answered와 committed를 찾아 보세요. 답을 계산했지만 저장하지 못한 경우까지 기록해야 재전송해도 될지 판단할 수 있습니다.

선수 개념 · 68장 FastAPI 요청 모델과 POST 엔드포인트

#### HTTP 200에서 실제 답과 출처 ID까지 내려가기

```mermaid
flowchart TD
nhttp["전송 기록"]
  nbody["python_response"]
  nresult["답변 결과"]
  nid["첫 출처"]
  nstored["저장 상태"]
  nhttp --> nbody
  nbody --> nresult
  nresult --> nid
  nbody --> nstored
```

저장된 전송 예시의 겉 상태와 안쪽 값을 읽는 경로입니다. HTTP 상태 코드만으로 인용한 문서까지 알 수는 없습니다.

## 답변 생성과 저장 성공 여부 구분

응답 예시　대화 lesson / 요청 unsaved / 로컬 전송 상태 503 / 업무 답 장비 B의 고정 장치 / 오류 코드 storage_failure. 이 경우 “관련 문서가 없습니다”라고 표시하면 다음 행동을 고르기 어렵습니다. 이전 요청이 저장되었는지 먼저 확인하도록 안내합니다.


### 팀 참고 예시 · 답변 생성 후 저장 실패


In [3]:
failed = {'request_id': 'unsaved',
 'session_id': 'lesson',
 'boundary_status': 'storage_error',
 'workflow_status': 'completed',
 'result': {'status': 'answered',
            'answer': '점검 전 고정 장치를 확인합니다.',
            'sources': [{'id': 'RULE-B',
                         'text': '점검 전 고정 장치를 확인합니다.',
                         'source': '가상 장비실 규정 B',
                         'page': 2,
                         'section': '2.2',
                         'revised_on': '2026-11-01',
                         'version': 'rules-v1'}],
            'mode': 'local_rule_fixture',
            'model_calls': 0},
 'storage_status': 'failed',
 'error': {'code': 'storage_failure',
           'stage': 'session_commit',
           'error_type': 'OSError'}}
print(failed["result"]["answer"])
print(failed["error"]["code"])

점검 전 고정 장치를 확인합니다.
storage_failure


고정 장치 답과 storage_failure가 함께 남습니다. 503이라고 해서 계산 결과까지 사라진 것은 아닙니다.


## API 소비자가 전송 실패와 답변 유보를 다르게 처리하는지 시험하기

준비 계약에서 Q01 응답은 HTTP 200이고 저장까지 마쳤습니다. 반면 unsaved는 계산한 답이 있어도 저장 오류로 503을 보냅니다. 화면 개발자가 이 두 응답을 실제 HTTP 클라이언트로 읽는 작은 계약 시험을 만듭니다.

### 셀 05 · 같은 클라이언트로 저장 실패 응답 읽기

이 셀은 서버의 저장 기능을 다시 만들지 않습니다. 제공된 실패 응답을 httpx의 테스트 전송 계층으로 보내, HTTP 상태와 본문에 동시에 접근하는 클라이언트를 시험합니다.

In [4]:
import httpx

def fixture_response(http_request):
    incoming = json.loads(http_request.content)
    if incoming["request_id"] == failed["request_id"]:
        return httpx.Response(503, json={**transport_row["body"], "python_response": failed})
    return httpx.Response(200, json=transport_row["body"])

with httpx.Client(transport=httpx.MockTransport(fixture_response),
                  base_url="https://lesson.invalid") as client:
    outgoing = {**request, "request_id": "unsaved",
                "question": "장비 B의 점검 전 확인 사항은?"}
    response = client.post("/p1/ask", json=outgoing)
message = response.json()["python_response"]
print(response.status_code, message["result"]["sources"][0]["id"])
print(message["storage_status"], message["error"]["stage"])

503 RULE-B
failed session_commit


503 본문에 RULE-B가 있고 저장 상태는 failed입니다. 예외 처리에서 HTTP 코드만 보고 본문을 통째로 버리면, 사용자에게 꼭 알려야 할 “답은 계산됐지만 저장하지 못했다”는 정보도 사라집니다.

### 팀 API의 계약 시험으로 바꾸기

정상 request로 같은 클라이언트를 호출해 200과 committed를 확인하세요. 다음에는 fixture_response 대신 팀의 로컬 서버에 같은 요청을 보내기 전에 세션과 접근 주체를 준비합니다. 위 시험은 URL에 인터넷 요청을 보내지 않습니다.

<mark>서비스 응답의 실패 단계는 HTTP 번호와 업무 본문을 함께 읽어야 알 수 있습니다.</mark> 저장 실패 뒤 검색을 반복해도 저장소의 문제가 해결되지는 않습니다. 오류의 stage가 session_commit인지부터 확인합니다.

103장에서는 같은 오류를 동료가 화면에서 읽은 후 어떤 다음 행동을 고르는지 관찰합니다.

## 팀 사례에 적용하기 · 팀의 조회 기록을 응답 계약에 붙이기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
matched = [doc for doc in incoming["prepared"] if doc["id"] == incoming["outcome"]]
reply = {"request_id": incoming["request_id"], "status": "answered" if matched else "no_evidence",
         "sources": matched, "question": incoming["question"]}
new = {"quality_reference": incoming["quality_reference"], "reply": reply, "adapter": incoming["adapter"], "checks": incoming["checks"]}
print(reply["request_id"], reply["status"], [doc["id"] for doc in matched])

team-99 answered ['RULE-A']


공개 조회에 근거가 있으면 ID를 반환합니다. 같은 계약에서 자료가 없는 경우를 시험할 때에는 빈 sources와 업무 상태를 함께 읽으세요.

In [ ]:
output_file = project_root / "team-project/final/chapter102.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter102.json


103장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

103장에는 정상 Q01과 저장 실패 요청의 상태·출처를 함께 넘깁니다. 이 장은 요청과 로컬 전송 자료를 노트북에서 읽습니다. 실제 HTTP 응답, DB와 인증 결과는 이 실행으로 확인할 수 없습니다.
